In [100]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
lim_fp = 0.92

In [101]:
df_pot = pd.read_excel('/content/Leitura_Medicao_Potencias.xlsx')

In [102]:
tempo = df_pot['Tempo']
pot_ativa = df_pot['Ativa']
pot_reativa = df_pot['Reativa']

In [103]:
pot_aparente = np.sqrt(pot_ativa**2 + pot_reativa**2)
sinal = pot_reativa / np.abs(pot_reativa)
fator_desloc = sinal * pot_ativa / pot_aparente
pot_aparente_092 = pot_ativa / 0.92
pot_reativa_092 = np.sqrt(pot_aparente_092**2 - pot_ativa**2)
pot_reativa_excedente = np.zeros(len(pot_reativa_092))

for ii in range(len(pot_reativa_092)):
    if pot_reativa[ii] >=0:
        pot_reativa_excedente[ii] = pot_reativa[ii] - pot_reativa_092[ii]
    else:
        pot_reativa_excedente[ii] = pot_reativa[ii] + pot_reativa_092[ii]


p_capacitores = - pot_reativa_excedente

In [104]:
delta_S = pot_aparente - pot_aparente_092
delta_P = delta_S * 0.92
delta_Q = np.sqrt(delta_S**2 - delta_P**2)

In [105]:
# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])
# Add traces
fig.add_trace(
    go.Bar(x=tempo, y=pot_ativa, name="Ativa"),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=tempo, y=pot_reativa, name="Reativa"),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=tempo, y=fator_desloc, name="Fator de Potência"),
    secondary_y=True,
)
# Add figure title
fig.update_layout(
    title_text=""
)
# Set x-axis title
fig.update_xaxes(title_text="Dia | Hora:Minuto")
# Set y-axes titles
fig.update_yaxes(title_text="Potências Ativa e Reativa", secondary_y=False)
fig.update_yaxes(title_text="Fator de Potência", secondary_y=True)
fig.show()

In [106]:
from datetime import datetime
tempo_horas = df_pot.Tempo.dt.hour.values
tempo_dia = df_pot.Tempo.dt.day.values
nn = len(tempo)
nn/24

7.0

In [107]:
indice_fator_desloc_negativo = np.where(fator_desloc<0.92)
indice_fator_desloc_positivo = np.where(fator_desloc>=0.92)

In [108]:
intervalo_de_inegralizacao = 1.0
energia_reativa_acumudada_indutiva = 0.0
energia_reativa_acumudada_capacitiva = 0.0
demanda_reativa_maxima_capacitiva = 0
demanda_reativa_maxima_indutiva = 0
instantes_excedente = []
dias_excedente = []
tempos_absoluto_excedente = []
reativo_excedente_kw = []
k_excedente = []
for k in range(nn):
    if tempo_horas[k]>=0 and tempo_horas[k]<=5: # madrugada
        if fator_desloc[k] < -0.0:
            if np.abs(fator_desloc[k]) < 0.92:
                energia_reativa_acumudada_capacitiva = energia_reativa_acumudada_capacitiva + np.abs(pot_reativa[k]) * intervalo_de_inegralizacao
                if demanda_reativa_maxima_capacitiva < np.abs(pot_reativa[k]):
                    demanda_reativa_maxima_capacitiva = np.abs(pot_reativa[k])
                    instantes_excedente.append(tempo_horas[k])
                    dias_excedente.append(tempo_dia[k])
                    tempos_absoluto_excedente.append(tempo[k])
                    reativo_excedente_kw.append(delta_P[k])
                    k_excedente.append(k)
     
    else: #dia
        if fator_desloc[k] >= 0.0:
            if fator_desloc[k] < 0.92:
                energia_reativa_acumudada_indutiva = energia_reativa_acumudada_indutiva + pot_reativa[k] * intervalo_de_inegralizacao
                if demanda_reativa_maxima_indutiva < pot_reativa[k]:
                    demanda_reativa_maxima_indutiva = pot_reativa[k]
                    instantes_excedente.append(tempo_horas[k])
                    dias_excedente.append(tempo_dia[k])
                    tempos_absoluto_excedente.append(tempo[k])
                    reativo_excedente_kw.append(delta_P[k])
                    k_excedente.append(k)



In [109]:
data = {'aaaa-mm-dd hh:mm:ss': tempos_absoluto_excedente, 
        'dia_nr': dias_excedente, 
        'hora_nr': instantes_excedente,
        'k_excedente': k_excedente,
        'Pot Ativa': pot_ativa[k_excedente],
        'Pot Reat': pot_reativa[k_excedente],
        'Fat.Pot.': fator_desloc[k_excedente],
        'Pot Cap': p_capacitores[k_excedente],
        'kw_exced': reativo_excedente_kw
        }

df_excedente = pd.DataFrame(data)

In [110]:
# Create figure with secondary y-axis
fig_excedente = make_subplots(specs=[[{"secondary_y": True}]])
# Add traces
fig_excedente.add_trace(
    go.Bar(x=df_excedente['aaaa-mm-dd hh:mm:ss'], y=df_excedente['Pot Reat'], name="Reativos Originais"),
    secondary_y=False,
)

fig_excedente.add_trace(
    go.Bar(x=df_excedente['aaaa-mm-dd hh:mm:ss'], y=df_excedente['Pot Cap'], name="Reativos a Gerar"),
    secondary_y=False,
)

fig_excedente.add_trace(
    go.Scatter(x=df_excedente['aaaa-mm-dd hh:mm:ss'], y=df_excedente['Fat.Pot.'], name="Fator de Potência", line=dict(width=1.5)),
    secondary_y=True,
)

# Add figure title
fig_excedente.update_layout(
    title_text="Momentos de baixo fator de potência"
)

fig_excedente.update_layout(
    yaxis2=dict(
        title='Secondary Y Axis Title',
        overlaying='y',
        side='right',
        range=[-1, 1]
    )
)


# Set x-axis title
fig_excedente.update_xaxes(title_text="Dia | Hora:Minuto")
# Set y-axes titles
fig_excedente.update_yaxes(title_text="Potências Ativa e Reativa", secondary_y=False)
fig_excedente.update_yaxes(title_text="Fator de Potência", secondary_y=True)
fig_excedente.show()

In [111]:
df_excedente

,aaaa-mm-dd hh:mm:ss,dia_nr,hora_nr,k_excedente,Pot Ativa,Pot Reat,Fat.Pot.,Pot Cap,kw_exced
0,2022-02-01 00:00:00,1,0,0,7,-7,-0.707107,4.018012,2.107535
6,2022-02-01 06:00:00,1,6,6,48,63,0.606043,-42.552086,24.866091
11,2022-02-01 11:00:00,1,11,11,90,93,0.695421,-54.660161,29.064493
13,2022-02-01 13:00:00,1,13,13,99,94,0.725182,-51.826177,26.596006
24,2022-02-02 00:00:00,2,0,24,6,-8,-0.600000,5.444011,3.200000
47,2022-02-02 23:00:00,2,23,47,24,96,0.242536,-85.776043,67.038172
61,2022-02-03 13:00:00,3,13,61,20,98,0.199960,-89.480036,72.018398
136,2022-02-06 16:00:00,6,16,136,18,99,0.178885,-91.332032,74.573214


In [115]:
acum_kw = np.cumsum( np.array(reativo_excedente_kw) )
acum_kw[-1]

299.46391061708334

In [113]:
# Create figure with secondary y-axis
fig_kw = make_subplots(specs=[[{"secondary_y": True}]])
# Add traces
fig_kw.add_trace(
    go.Bar(x=df_excedente['aaaa-mm-dd hh:mm:ss'], y=df_excedente['kw_exced'], name="Integralizado"),
    secondary_y=False,
)

fig_kw.add_trace(
    go.Scatter(x=df_excedente['aaaa-mm-dd hh:mm:ss'], y=acum_kw, name="Acumulado"),
    secondary_y=True,
)



# Add figure title
fig_kw.update_layout(
    title_text="Reativos excedidos em kWh equivalente para um fator de potência 0,92"
)




# Set x-axis title
fig_kw.update_xaxes(title_text="Dia | Hora:Minuto")
# Set y-axes titles
fig_kw.update_yaxes(title_text="Reativos excedidos integralizados", secondary_y=False)
fig_kw.update_yaxes(title_text="Reativos excedidos em kWh acumulados", secondary_y=True)
fig_kw.show()